In [1]:
import numpy as np
import torch

In [2]:
X = np.array([160, 170, 180, 190])
y = np.array([0, 0, 1, 1])
X, y

(array([160, 170, 180, 190]), array([0, 0, 1, 1]))

In [3]:
X_mean = np.mean(X)
X_std = np.std(X)
X_norm = (X-X_mean) / X_std
X_norm

array([-1.34164079, -0.4472136 ,  0.4472136 ,  1.34164079])

In [4]:
X_norm_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# torch.nn.Linear(1, 1)에 넣으려면 각 데이터가 입력 특성 1개를 가진 형태, shape(n, 1)이어야 함
# reshape로 2차원으로, 1열로 모양을 바꿔줌
print(X_norm_tensor, y_tensor)
X_norm_tensor = X_norm_tensor.reshape(-1, 1)
y_tensor = y_tensor.reshape(-1, 1)
X_norm_tensor, y_tensor

tensor([-1.3416, -0.4472,  0.4472,  1.3416]) tensor([0., 0., 1., 1.])


(tensor([[-1.3416],
         [-0.4472],
         [ 0.4472],
         [ 1.3416]]),
 tensor([[0.],
         [0.],
         [1.],
         [1.]]))

In [5]:
# a, b를 직접 만들지 않고 linear생성
# pytorch의 랜덤 초기값 고정
# 고정해야 실행할때마다 결과가 크게 달라지지 않음
torch.manual_seed(42)

# 입력특성 1개를 받아 출력 1개로 내보내기
# 내부적으로 H(x) = a * X_norm + b 계산
linear =torch.nn.Linear(1, 1)

# a, b 초기값 확인
print('linear.weight:', linear.weight)
print('linear.bias:', linear.bias)

linear.weight: Parameter containing:
tensor([[0.7645]], requires_grad=True)
linear.bias: Parameter containing:
tensor([0.8300], requires_grad=True)


In [6]:
# binary cross entropy cost 계산하기
criterion = torch.nn.BCELoss()

print('criterion 준비 완료:', criterion)

criterion 준비 완료: BCELoss()


In [7]:
# 학습 설정
learning_rate = 0.1
epochs = 1000

learning_rate, epochs


(0.1, 1000)

In [8]:
optimizer = torch.optim.SGD(linear.parameters(), lr=learning_rate)
list(linear.parameters())

[Parameter containing:
 tensor([[0.7645]], requires_grad=True),
 Parameter containing:
 tensor([0.8300], requires_grad=True)]

In [9]:
# nn.linear + nn.bceloss로 경사 하강법 학습

for epoch in range(epochs):
    # 0 초기화
    optimizer.zero_grad()

    # 선형식 계산
    H = linear(X_norm_tensor)

    # sigmoid 적용 예측 확률 계산
    z = torch.sigmoid(H)

    # Cost 계산
    mean_cost = criterion(z, y_tensor)

    # grad 자동 계산
    mean_cost.backward()

    # weight, bias 업데이트
    optimizer.step()

    if epoch%100 == 0 or epoch == epochs - 1:
        print(
            f'epoch={epoch}, '
            f'Cost={mean_cost.item():.6f}'
            f'weight(a)={linear.weight.item():.6f}'
            f'bias(b)={linear.bias.item():.6f}'
        )
    if epoch < 3:
        print(
            f'  (확인용) linear.weight.grad={linear.weight.grad.item():.6f}'
            f'linear.bias.grad={linear.bias.grad.item():.6f}'
        )

epoch=0, Cost=0.495464weight(a)=0.793780bias(b)=0.812529
  (확인용) linear.weight.grad=-0.292415linear.bias.grad=0.174793
  (확인용) linear.weight.grad=-0.286153linear.bias.grad=0.169918
  (확인용) linear.weight.grad=-0.280072linear.bias.grad=0.165171
epoch=100, Cost=0.178670weight(a)=2.290082bias(b)=0.173212
epoch=200, Cost=0.125357weight(a)=3.002210bias(b)=0.061586
epoch=300, Cost=0.099283weight(a)=3.509002bias(b)=0.026837
epoch=400, Cost=0.082901weight(a)=3.912263bias(b)=0.013229
epoch=500, Cost=0.071398weight(a)=4.250606bias(b)=0.007116
epoch=600, Cost=0.062789weight(a)=4.543496bias(b)=0.004091
epoch=700, Cost=0.056068weight(a)=4.802371bias(b)=0.002480
epoch=800, Cost=0.050660weight(a)=5.034644bias(b)=0.001570
epoch=900, Cost=0.046207weight(a)=5.245449bias(b)=0.001031
epoch=999, Cost=0.042507weight(a)=5.436657bias(b)=0.000701


In [10]:
# weight(a), bias(b) 확인
print('학습된 weight(a):', linear.weight.item())
print('학습된 weight(b):', linear.bias.item())

print('linear.weight:', linear.weight)
print('lienar.bias:', linear.bias)

학습된 weight(a): 5.436656951904297
학습된 weight(b): 0.0007013267604634166
linear.weight: Parameter containing:
tensor([[5.4367]], requires_grad=True)
lienar.bias: Parameter containing:
tensor([0.0007], requires_grad=True)


In [ ]:
# 새로운 입력값 예측
input_height = 185
input_norm = (input_height - X_mean) / X_std

with torch.no_grad():
    # (1,1)로 shape 맞추기
    input_norm_tensor = torch.tensor([[input_norm]], dtype=torch.float32)
    print('input_norm_tensor shape:', input_norm_tensor.shape)

    H_new = linear(input_norm_tensor)
    z_new = torch.sigmoid(H_new)
    pred = 1 if z_new.item() >= 0.5 else 0

    print(f'키가 {input_height}cm인 사람의 농구선수 확률(z): {z_new.item():.4f}')
    if pred == 1:
        print('판별 결과: 농구선수입니다.')
    else:
        print('판별 결과: 농구선수가 아닙니다.')

input_norm_tensor shape: torch.Size([1, 1])
키가 185cm인 사람의 농구선수 확률(z): 0.9923
판별 결과: 농구선수입니다.
